# Stock Markets Analytics Zoomcamp 2026 — Module 2 Homework: One Dataframe

Run each code cell, then fill in the **Answer** markdown cell below it.

Requirements: `pip install pandas numpy yfinance requests lxml gdown pyarrow`

Notes on data sources (confirmed against the actual course spec + lecture notebook):
- Q1 uses `iposcoop.com/ipos-recently-filed/` (the Withdrawn/Postponed list) —
  a **different** page from the lecture's `stockanalysis.com` IPO source.
- Q2/Q3 use `iposcoop.com/2025-pricings/` — also different from the lecture's
  `get_ipos_by_year()` (which pulls stockanalysis.com), per the homework spec.
- Q4 uses a separate precomputed parquet file (via `gdown`), with RSI as a
  **lowercase** `rsi` column — matching this course's TA-Lib convention.


In [ ]:
import io
import re
import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}


## Question 1: [IPO] Withdrawn IPOs by Company Type

What is the total withdrawn IPO value (in $ millions) for the company class
with the highest total withdrawal value?

- 200
- 300
- 400
- 500

Source: https://www.iposcoop.com/ipos-recently-filed/ — filter to
'Expected To Trade' == 'Withdrawn' (should be 32 entries).


In [ ]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
resp = requests.get(url, headers=HEADERS)
tables = pd.read_html(io.StringIO(resp.text))

ipo_df = None
for t in tables:
    if any("Expected" in str(c) for c in t.columns):
        ipo_df = t
        break

print("Columns found:", list(ipo_df.columns))
withdrawn = ipo_df[ipo_df["Expected To Trade"].astype(str).str.contains("Withdrawn", case=False, na=False)].copy()
print(f"Withdrawn entries: {len(withdrawn)}  (expected 32)")
withdrawn.head()


In [ ]:
def classify_company(name):
    name = str(name)
    if "Technologies" in name:
        return "Technologies"
    if "Acquisition Corp" in name or "Acquisition Corporation" in name or "Corp" in name:
        return "Acquisition Corp"
    if "Inc" in name or "Incorporated" in name:
        return "Inc."
    if "Group" in name:
        return "Group"
    if "Ltd" in name or "Limited" in name:
        return "Limited"
    if "Holdings" in name or "Holding" in name:
        return "Holdings"
    return "Other"

name_col = [c for c in withdrawn.columns if "Company" in str(c)][0]
withdrawn["Company Type"] = withdrawn[name_col].apply(classify_company)

def parse_price(val):
    if pd.isna(val) or str(val).strip() in ("-", ""):
        return None
    nums = re.findall(r"[\d.]+", str(val))
    nums = [float(n) for n in nums]
    if not nums:
        return None
    return sum(nums) / len(nums)

price_col = [c for c in withdrawn.columns if "Price" in str(c)][0]
withdrawn["Avg_price"] = withdrawn[price_col].apply(parse_price)

def to_numeric_clean(val):
    if pd.isna(val) or str(val).strip() in ("-", ""):
        return np.nan
    cleaned = re.sub(r"[$,]", "", str(val)).strip()
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

shares_col = [c for c in withdrawn.columns if "Shares" in str(c)][0]
vol_col = [c for c in withdrawn.columns if "Vol" in str(c)][0]
withdrawn["Shares_millions"] = withdrawn[shares_col].apply(to_numeric_clean)
withdrawn["Est_Vol_millions"] = withdrawn[vol_col].apply(to_numeric_clean)

computed_value = withdrawn["Shares_millions"] * withdrawn["Avg_price"]
withdrawn["Shares_offered_value"] = computed_value.where(
    computed_value.notna(), withdrawn["Est_Vol_millions"]
)

result = withdrawn.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)
print(result)
print(f"\nHighest: {result.index[0]} with ${result.iloc[0]:.1f}M")


### Answer 1

- Company type with highest total withdrawal value: **`<fill in>`**
- Total value ($M): **`<fill in>`**


## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

What is the median Sharpe ratio (as of 11 September 2026) for companies
that went public before 1 September 2025?

- -0.04
- 0.04
- 0.1
- 0.2

Source: https://www.iposcoop.com/2025-pricings/ — filter Offer Date <
2025-09-01, exclude 0% return (should leave 148 stocks; ~134 after
yfinance download, since some may be delisted).


In [ ]:
url2 = "https://www.iposcoop.com/2025-pricings/"
resp2 = requests.get(url2, headers=HEADERS)
tables2 = pd.read_html(io.StringIO(resp2.text))

ipo_2025 = None
for t in tables2:
    if any("Offer" in str(c) for c in t.columns):
        ipo_2025 = t
        break

print("Columns:", list(ipo_2025.columns))
print(f"Total rows: {len(ipo_2025)}")


In [ ]:
date_col = [c for c in ipo_2025.columns if "Offer" in str(c) and "Date" in str(c)][0]
ipo_2025[date_col] = pd.to_datetime(ipo_2025[date_col], errors="coerce")

return_col = [c for c in ipo_2025.columns if "Return" in str(c) or "%" in str(c)]
return_col = return_col[0] if return_col else None

filtered = ipo_2025[ipo_2025[date_col] < "2025-09-01"].copy()
if return_col:
    filtered = filtered[filtered[return_col].astype(str).str.strip() != "0.00%"]

print(f"Filtered IPOs (before Sep 1 2025, non-zero return): {len(filtered)}  (expected ~148)")


In [ ]:
ticker_col = [c for c in filtered.columns if "Symbol" in str(c) or "Ticker" in str(c)][0]
tickers = filtered[ticker_col].dropna().unique().tolist()
print(f"Tickers to download: {len(tickers)}")

all_data = []
failed = []
for tkr in tickers:
    try:
        data = yf.download(tkr, start="2024-01-01", end="2026-09-12",
                            progress=False, auto_adjust=False)
        if data.empty:
            failed.append(tkr)
            continue
        data = data.copy()
        data["Close"] = data["Close"].squeeze()  # handle yfinance MultiIndex columns
        data["ticker"] = tkr
        all_data.append(data)
    except Exception:
        failed.append(tkr)

print(f"Successfully downloaded: {len(all_data)}  (expected ~134)")
print(f"Failed/delisted: {len(failed)}")

stocks_df = pd.concat(all_data).reset_index()


In [ ]:
stocks_df = stocks_df.sort_values(["ticker", "Date"])
stocks_df["growth_252d"] = stocks_df.groupby("ticker")["Close"].transform(lambda x: x / x.shift(252))
stocks_df["volatility"] = stocks_df.groupby("ticker")["Close"].transform(
    lambda x: x.rolling(30).std() * np.sqrt(252)
)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

snapshot = stocks_df[stocks_df["Date"] == "2026-09-11"]
print(f"Stocks with data on 2026-09-11: {len(snapshot)}")
print(snapshot[["growth_252d", "volatility", "Sharpe"]].describe())

median_sharpe = snapshot["Sharpe"].median()
print(f"\nMEDIAN SHARPE RATIO: {median_sharpe:.3f}")


### Answer 2

- Median Sharpe ratio (2026-09-11): **`<fill in>`**


## Question 3: [IPO] 'Fixed Months Holding Strategy'

What is the optimal number of months (1 to 12) to hold a newly IPO'd stock
in order to maximize the median growth value?

- 1
- 3
- 5
- 7

Uses `stocks_df` from Question 2. 1 month = 21 trading days.


In [ ]:
for m in range(1, 13):
    days = m * 21
    stocks_df[f"future_growth_{m}_m"] = stocks_df.groupby("ticker")["Close"].transform(
        lambda x: x.shift(-days) / x
    )

min_dates = stocks_df.groupby("ticker")["Date"].min().reset_index()
min_dates.columns = ["ticker", "min_date"]

entry_df = stocks_df.merge(min_dates, on="ticker").query("Date == min_date")

growth_cols = [f"future_growth_{m}_m" for m in range(1, 13)]
desc = entry_df[growth_cols].describe()
print(desc)

medians = desc.loc["50%"]
best_month = medians.idxmax()
print(f"\nBest holding period: {best_month} -> median growth {medians.max():.4f}")


### Answer 3

- Optimal holding period (months): **`<fill in>`**
- Corresponding max median growth: **`<fill in>`**


## Question 4: [Strategy] Simple RSI-Based Trading Strategy

What is the total profit (in $ thousands) you would have earned by
investing $1000 every time a stock was oversold (RSI < 30)?

- 65
- 85
- 105
- 125

Uses a precomputed parquet file with technical/macro indicators for a
broad set of tickers, filtered 2000-01-01 to 2025-06-01. Note: RSI column
is lowercase `rsi`, matching this course's TA-Lib convention (confirmed
from the Module 2 lecture notebook's `talib_get_momentum_indicators_for_one_ticker`
function, which outputs a lowercase `rsi` column).


In [ ]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")
print(df.shape)
print(df.columns.tolist())


In [ ]:
# Find the actual RSI and Date column names in this file (case can vary)
rsi_col = [c for c in df.columns if c.lower() == "rsi"][0]
date_col = "Date" if "Date" in df.columns else df.index.name
print(f"Using RSI column: '{rsi_col}', Date column/index: '{date_col}'")

if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col])
else:
    df.index = pd.to_datetime(df.index)
    df = df.reset_index()
    date_col = df.columns[0]

selected_df = df[
    (df[rsi_col] < 30) &
    (df[date_col] >= "2000-01-01") &
    (df[date_col] <= "2025-06-01")
].copy()

print(f"Number of RSI<30 signals: {len(selected_df)}  (expected ~5,206)")

growth_col = [c for c in df.columns if "growth_future_30d" in c.lower()][0]
net_income = 1000 * (selected_df[growth_col] - 1).sum()
avg_return = (selected_df[growth_col] - 1).mean()
win_rate = (selected_df[growth_col] > 1).mean()

print(f"Net income: ${net_income:,.0f}  (${net_income/1000:.1f}K)")
print(f"Average 30-day return: {avg_return*100:.2f}%  (expected ~1.26%)")
print(f"Win rate: {win_rate*100:.2f}%  (expected ~55.13%)")


### Answer 4

- Number of RSI<30 signals found: **`<fill in>`**
- Net income ($K): **`<fill in>`**


## Question 5 (Optional): Predicting a Positive-Return IPO

Most IPO strategies deliver negative average/median returns (even the 75th
percentile). How would you change the strategy to increase profitability?


### Answer 5

`<fill in your ideas here>`
